# Notebook 4: Gold Layer - Agrégats Métier

**Objectif**: Créer des vues agrégées prêtes pour l'analyse et la reporting.

## Qu'est-ce que la couche Gold?

La **couche Gold** est la couche de présentation du lakehouse:
- **Agrégats**: Données pré-calculées pour les performances
- **KPIs**: Indicateurs clés de performance
- **Optimisé pour le reporting**: Requêtes rapides pour dashboards

## Flux de données

```
Silver (clean) ──► Agrégation ──► Gold (business ready)
```

---
## 1. Initialisation

In [ ]:
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark

spark = get_spark("gold-aggregations")

print("✅ Session Spark initialisée")

## 2. Lecture de la table Silver

La couche Gold lit depuis Silver (données déjà nettoyées).

In [ ]:
# Lecture de la table Silver
sales_silver = spark.table("nessie.silver.sales")

silver_count = sales_silver.count()
print(f"📊 Enregistrements dans Silver: {silver_count:,}")

# Aperçu
print("\n=== Aperçu Silver ===")
sales_silver.select("order_id", "category", "region", "sales", "profit").show(5, truncate=False)

## 3. Création de l'agrégat: Ventes par Catégorie et Région

**KPIs calculés**:
- `total_sales`: Somme des ventes
- `total_profit`: Somme des profits
- `total_quantity`: Quantité totale vendue
- `order_count`: Nombre de commandes

Cet agrégat permet d'analyser les performances par combinaison Catégorie × Région.

In [ ]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round, count

# Agrégation par catégorie et région
sales_by_category_region = (
    sales_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

print("=== Agrégat: Ventes par Catégorie × Région ===")
sales_by_category_region.orderBy("total_sales", ascending=False).show(truncate=False)

## 4. Création d'autres agrégats utiles

In [ ]:
# Agrégat: Ventes par Segment (Client, Corporate, Home Office)
sales_by_segment = (
    sales_silver
    .groupBy("segment")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_round(spark_avg("sales"), 2).alias("avg_order_value"),
        count("order_id").alias("order_count")
    )
)

print("=== Agrégat: Ventes par Segment ===")
sales_by_segment.orderBy("total_sales", ascending=False).show(truncate=False)

In [ ]:
# Agrégat: Top produits
top_products = (
    sales_silver
    .groupBy("category", "sub_category", "product_name")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity")
    )
)

print("=== Top 10 Produits ===")
top_products.orderBy("total_sales", ascending=False).show(10, truncate=False)

## 5. Suppression des anciennes tables Gold (si elles existent)

In [ ]:
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_category_region")
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_segment")
spark.sql("DROP TABLE IF EXISTS nessie.gold.top_products")

print("✅ Anciennes tables Gold supprimées")

## 6. Création des tables Gold

In [ ]:
# Créer les tables Gold
sales_by_category_region.writeTo("nessie.gold.sales_by_category_region").using("iceberg").create()
sales_by_segment.writeTo("nessie.gold.sales_by_segment").using("iceberg").create()
top_products.writeTo("nessie.gold.top_products").using("iceberg").create()

print("✅ Tables Gold créées:")
print("   • nessie.gold.sales_by_category_region")
print("   • nessie.gold.sales_by_segment")
print("   • nessie.gold.top_products")

## 7. Vérification des tables Gold

In [ ]:
# Lister toutes les tables Gold
print("=== Tables dans Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

In [ ]:
# Vérifier les enregistrements
print("=== Nombre d'enregistrements ===")
spark.sql("SELECT 'sales_by_category_region' as table_name, COUNT(*) as count FROM nessie.gold.sales_by_category_region").show()
spark.sql("SELECT 'sales_by_segment' as table_name, COUNT(*) as count FROM nessie.gold.sales_by_segment").show()
spark.sql("SELECT 'top_products' as table_name, COUNT(*) as count FROM nessie.gold.top_products").show()

## 8. Exemples de requêtes analytiques

Les tables Gold sont optimisées pour les requêtes métier.

In [ ]:
# Top 5 par ventes
print("=== Top 5 Catégorie × Région ===")
spark.sql("""
    SELECT category, region, total_sales, total_profit, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

In [ ]:
# Meilleure marge par région
print("=== Marge bénéficiaire par région ===")
spark.sql("""
    SELECT 
        region,
        SUM(total_sales) as sales,
        SUM(total_profit) as profit,
        ROUND(100.0 * SUM(total_profit) / SUM(total_sales), 2) as margin_pct
    FROM nessie.gold.sales_by_category_region
    GROUP BY region
    ORDER BY profit DESC
""").show(truncate=False)

In [ ]:
# Performance par catégorie
print("=== Performance par catégorie ===")
spark.sql("""
    SELECT 
        category,
        SUM(total_sales) as global_sales,
        SUM(total_profit) as global_profit,
        SUM(order_count) as global_orders
    FROM nessie.gold.sales_by_category_region
    GROUP BY category
    ORDER BY global_sales DESC
""").show(truncate=False)

## 9. Résumé du Lakehouse complet

In [ ]:
# Récupérer les counts de chaque couche
bronze_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.bronze.sales").first()[0]
silver_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.silver.sales").first()[0]
gold_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.gold.sales_by_category_region").first()[0]

print("""
╔══════════════════════════════════════════════════════════════╗
║              LAKEHOUSE COMPLET - RÉSUMÉ                     ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  Architecture Medallion:                                   ║
║                                                            ║
║  ┌─────────┐     ┌─────────┐     ┌─────────┐              ║
║  │ BRONZE  │────►│ SILVER  │────►│   GOLD   │              ║
║  │  Raw    │     │  Clean  │     │  Aggregates│           ║
║  └─────────┘     └─────────┘     └─────────┘              ║
║    {bronze_count:,}           {silver_count:,}             {gold_count:,}                 ║
║                                                            ║
║  Tables créées:                                           ║
║    Bronze:  nessie.bronze.sales                           ║
║    Silver:  nessie.silver.sales                           ║
║    Gold:    nessie.gold.sales_by_category_region           ║
║             nessie.gold.sales_by_segment                   ║
║             nessie.gold.top_products                       ║
║                                                            ║
║  Technologies:                                             ║
║    • Apache Spark (traitement)                            ║
║    • Apache Iceberg (format de table ACID)                ║
║    • Project Nessie (catalogue versionné)                 ║
║    • AWS S3 (stockage)                                    ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")

print("\n✅ Pipeline Bronze → Silver → Gold terminé avec succès!")
print("\n➡️ Prochaine étape: Exécuter '06_complete_demo.ipynb' pour la démo Nessie")